# Neural PagedAttention — marathon training (Colab)

**Runtime:** GPU (T4 or better; A100/L4 preferred for LLM training).  
**Before running:** push this repo from your machine, then set `REPO_URL` below (HTTPS or SSH).

This notebook:
1. Installs PyTorch + Unsloth + deps
2. Clones your repo
3. Logs in to Hugging Face (for `Qwen/Qwen2.5-1.5B-Instruct` and optional uploads)
4. Trains **DQN** for a long run (quality-first defaults)
5. Trains **LoRA policy gradient** (`train_ppo.py`) for a long run
6. Optionally zips artifacts for download or pushes LoRA to the Hub

Training time can be many hours; that is intentional.

In [ ]:
# --- Configure ---
REPO_URL = "https://github.com/YOUR_USER/NeuralPagedAttention.git"  # change me
# Use "" to clone the repo default branch (fixes many exit-128 'branch not found' errors).
BRANCH = "main"  # or "master", or "" for default
# Private repo: create a fine-grained PAT and set:
# REPO_URL = "https://YOUR_TOKEN@github.com/user/NeuralPagedAttention.git"
# (or set GITHUB_TOKEN and build URL in code — never commit tokens)

# Optional: paste a token from https://huggingface.co/settings/tokens (read for models, write if uploading)
HF_TOKEN = ""  # or set in Colab secrets as HF_TOKEN

import os
from google.colab import userdata
try:
    HF_TOKEN = userdata.get("HF_TOKEN") or HF_TOKEN
except Exception:
    pass

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

# Large cache on persistent disk if you use Colab persistent storage
os.environ.setdefault("HF_HOME", "/content/hf_cache")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

!nvidia-smi

In [ ]:
# --- Install stack (Colab-friendly) ---
# Unsloth pins a working CUDA stack; see https://github.com/unslothai/unsloth
!pip install -q --upgrade pip
!pip install -q "torch>=2.2" torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes transformers sentencepiece protobuf
!pip install -q numpy pydantic fastapi uvicorn huggingface_hub

import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# --- Clone repo (run Configure cell first: REPO_URL, BRANCH) ---
# Exit 128 fixes: wrong branch name (try BRANCH=None), private repo (GITHUB_TOKEN), or typo in URL.
import os
import subprocess

DEST = "/content/NeuralPagedAttention"
os.chdir("/content")
subprocess.run(["rm", "-rf", DEST], check=False)

def _clone(args: list[str]) -> subprocess.CompletedProcess:
    return subprocess.run(args, capture_output=True, text=True)

# Private GitHub: set GITHUB_TOKEN in Configure cell → use https://TOKEN@github.com/user/repo.git
cmd = ["git", "clone", "--depth", "1", REPO_URL, DEST]
if BRANCH and str(BRANCH).strip():
    cmd = ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, DEST]

r = _clone(cmd)
if r.returncode != 0:
    print("git stderr (first attempt):", r.stderr or r.stdout)
    print("Retrying without --branch (uses remote default, often fixes 'not found' for main vs master)...")
    subprocess.run(["rm", "-rf", DEST], check=False)
    r2 = _clone(["git", "clone", "--depth", "1", REPO_URL, DEST])
    if r2.returncode != 0:
        print("git stderr (fallback):", r2.stderr or r2.stdout)
        raise subprocess.CalledProcessError(r2.returncode, r2.args, r2.stdout, r2.stderr)

os.chdir(DEST)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)
subprocess.run(["git", "branch", "-a"], check=False)
print("Cloned OK:", DEST)


In [ ]:
# --- Hugging Face login (model download + optional push) ---
from huggingface_hub import login
import os
tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if tok:
    login(token=tok, add_to_git_credential=False)
else:
    login()  # interactive in Colab

print("HF login OK")

## 1) Marathon DQN (tabular-free, fast env steps)

Uses **`train_dqn.py` at repo root** so training works even if an old `dqn.py` still passes `--train` as a task name.

Defaults: **1000 episodes**, eval every **50**, full task horizons. Produces:
- `agents/NeuralAgent/dqn_weights.pth`
- `agents/NeuralAgent/dqn_weights_best.pth` (best mean score on easy+medium+hard eval)

In [ ]:
%cd /content/NeuralPagedAttention

!python train_dqn.py \
  --episodes 1000 \
  --eval-every 50 \
  --seed 42

## 2) Marathon LLM + LoRA (Unsloth policy gradient)

Defaults: **220 episodes**, full `max_ticks` per task (no artificial cap), tuned LR/entropy. Produces:
- `agents/UnslothPPOAgent/ppo_lora_agent/` (adapter + tokenizer)

If you hit OOM, re-run the next cell with `--max-ticks 1200` added to the command.

In [ ]:
%cd /content/NeuralPagedAttention

!python agents/UnslothPPOAgent/train_ppo.py \
  --episodes 220 \
  --lr 2e-5 \
  --entropy-coef 0.05

## 3) One-shot launcher (optional)

Same marathon as above, from `colab/marathon_train.py`.

In [ ]:
%cd /content/NeuralPagedAttention
!python colab/marathon_train.py --seed 42

## 4) Save artifacts (zip) or push LoRA to Hub

Replace `YOUR_NAME` with your Hub username and create the model repo first if pushing.

In [ ]:
%cd /content/NeuralPagedAttention

!zip -r /content/npa_artifacts.zip \
  agents/NeuralAgent/dqn_weights.pth \
  agents/NeuralAgent/dqn_weights_best.pth \
  agents/UnslothPPOAgent/ppo_lora_agent

from google.colab import files
files.download("/content/npa_artifacts.zip")

In [ ]:
# Optional: push LoRA folder to Hugging Face (create repo YOUR_NAME/npa-ppo-lora first)
HF_USER = "YOUR_NAME"
REPO_ID = f"{HF_USER}/npa-ppo-lora"

from huggingface_hub import HfApi
import os
api = HfApi(token=os.environ.get("HF_TOKEN"))
api.upload_folder(
    folder_path="/content/NeuralPagedAttention/agents/UnslothPPOAgent/ppo_lora_agent",
    repo_id=REPO_ID,
    repo_type="model",
)
print("Uploaded to", REPO_ID)